In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
import numpy as np

In [2]:
num_epochs = 10
batch_size = 100
learning_rate = 0.001

In [3]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [4]:
train_dataset = datasets.FashionMNIST('~/.pytorch/F_MNIST_data/',
                            train=True,
                            transform=transforms.ToTensor(),
                            download=True)

test_dataset = datasets.FashionMNIST('~/.pytorch/F_MNIST_data/',
                           train=False,
                           transform=transforms.ToTensor())

100%|██████████| 26.4M/26.4M [00:05<00:00, 4.41MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 126kB/s]
100%|██████████| 4.42M/4.42M [00:02<00:00, 2.06MB/s]
100%|██████████| 5.15k/5.15k [00:00<?, ?B/s]


In [5]:
train_loader = torch.utils.data.DataLoader(dataset=train_dataset,
                                           batch_size=batch_size,
                                           shuffle=True)

test_loader = torch.utils.data.DataLoader(dataset=test_dataset,
                                          batch_size=batch_size,
                                          shuffle=False)

In [6]:
class Teacher(nn.Module):
    def __init__(self):
        super(Teacher, self).__init__()
        self.layer1 = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=5, padding=2),
            nn.BatchNorm2d(16),
            nn.ReLU())
        self.layer2 = nn.Sequential(
            nn.Conv2d(16, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2))
        self.layer3 = nn.Sequential(
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2))
        self.fc1 = nn.Linear(7*7*32, 300)
        self.fc2 = nn.Linear(300, 10)

    def forward(self, x):
        out = self.layer1(x)
        out = self.layer2(out)
        out = self.layer3(out)
        out = out.view(out.size(0), -1)
        out = F.relu(self.fc1(out))
        out = self.fc2(out)
        out = F.log_softmax(out, dim=1)
        return out


In [7]:
class Student(nn.Module):
    def __init__(self):
        super(Student, self).__init__()
        self.layer1 = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2))
        self.fc1 = nn.Linear(14*14*16, 10)

    def forward(self, x):
        out = self.layer1(x)
        out = out.view(out.size(0), -1)
        out = self.fc1(out)
        return F.log_softmax(out, dim=1)

In [8]:
# The below function is called to reinitialize the weights of the network and define the required loss criterion and the optimizer.
def reset_model(is_teacher = True):
    if is_teacher == True:
        net = Teacher()
    else:
        net = Student()

    net = net.to(device)

    # Loss and Optimizer
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(net.parameters(), lr=learning_rate)
    return net, criterion, optimizer

In [9]:
teacher, criterion, optimizer = reset_model()

In [10]:
# The first step is to train the teacher network to become an expert. We move ahead with regular training procedure using the cross entropy loss and the Adam optimizer.

def training(net, reset = True):
    if reset == True:
        net, criterion, optimizer = reset_model()
    else:
        criterion = nn.CrossEntropyLoss()
        optimizer = torch.optim.Adam(net.parameters(), lr=learning_rate)

    net.train()
    for epoch in range(num_epochs):
        total_loss = 0
        accuracy = []
        for i, (images, labels) in enumerate(train_loader):
            images = images.to(device)
            labels = labels.to(device)
            temp_labels = labels


            # Forward + Backward + Optimize
            optimizer.zero_grad()
            outputs = net(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            correct = (predicted == temp_labels).sum().item()
            accuracy.append(correct/float(batch_size))

        print('Epoch: %d, Loss: %.4f, Accuracy: %.4f' %(epoch+1,total_loss, (sum(accuracy)/float(len(accuracy)))))

    return net

In [11]:
# Test the Model
def testing(net):
    net.eval()
    correct = 0
    total = 0
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)
        outputs = net(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    print('Test Accuracy of the network on the 10000 test images: %.2f %%' % (100.0 * correct / total))

In [12]:
reset = True
teacher = training(teacher, reset)
testing(teacher)

Epoch: 1, Loss: 231.3309, Accuracy: 0.8588
Epoch: 2, Loss: 155.7189, Accuracy: 0.9039
Epoch: 3, Loss: 131.6588, Accuracy: 0.9178
Epoch: 4, Loss: 113.8977, Accuracy: 0.9300
Epoch: 5, Loss: 103.2592, Accuracy: 0.9352
Epoch: 6, Loss: 89.8912, Accuracy: 0.9436
Epoch: 7, Loss: 79.5218, Accuracy: 0.9499
Epoch: 8, Loss: 68.9858, Accuracy: 0.9573
Epoch: 9, Loss: 60.6524, Accuracy: 0.9621
Epoch: 10, Loss: 52.4306, Accuracy: 0.9666
Test Accuracy of the network on the 10000 test images: 91.72 %


In [13]:
temperature = 1.5
for p in teacher.parameters():
    p.requires_grad= False

student, criterion, optimizer = reset_model(is_teacher = False)
alpha = 0.6

mse_criterion = nn.MSELoss()
softmax = nn.Softmax()

print(student)

Student(
  (layer1): Sequential(
    (0): Conv2d(1, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (fc1): Linear(in_features=3136, out_features=10, bias=True)
)


In [ ]:
# Training and testing the student network
# Train the Model

for epoch in range(num_epochs):
    total_loss = 0
    accuracy = []
    for i, (images, labels) in enumerate(train_loader):
        images = images.to(device)
        labels = labels.to(device)
        temp_labels = labels

        # Forward + Backward + Optimize
        optimizer.zero_grad()

        # student_outputs = student(images)

        # hard_outputs = teacher(images)
        # soft_outputs = hard_outputs/ temperature
        # soft_outputs = softmax(soft_outputs)

        # hard_loss = criterion(student_outputs, labels)
        # soft_loss = mse_criterion(student_outputs, soft_outputs)

        student_outputs = student(images)

        teacher_outputs = teacher(images)
        soft_outputs = teacher_outputs / temperature
        soft_outputs = softmax(soft_outputs)

        hard_loss = criterion(student_outputs, labels)
        soft_loss = mse_criterion(student_outputs, soft_outputs)

        loss = alpha*hard_loss + (1-alpha)*soft_loss
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        _, predicted = torch.max(student_outputs.data, 1)
        correct = (predicted == temp_labels).sum().item()
        accuracy.append(correct/float(batch_size))

    print('Epoch: %d, Loss: %.4f, Accuracy: %.4f' %(epoch+1,total_loss, (sum(accuracy)/float(len(accuracy)))))

c:\Learning\Labs\venv\lib\site-packages\torch\nn\modules\module.py:1773: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  return self._call_impl(*args, **kwargs)


Epoch: 1, Loss: 2068.6447, Accuracy: 0.8185
Epoch: 2, Loss: 2037.1333, Accuracy: 0.8658
Epoch: 3, Loss: 2031.0147, Accuracy: 0.8773
Epoch: 4, Loss: 2027.0660, Accuracy: 0.8850
Epoch: 5, Loss: 2024.0545, Accuracy: 0.8890
Epoch: 6, Loss: 2021.8782, Accuracy: 0.8927
Epoch: 7, Loss: 2020.2385, Accuracy: 0.8964
Epoch: 8, Loss: 2019.2415, Accuracy: 0.8979
Epoch: 9, Loss: 2018.0993, Accuracy: 0.8995
Epoch: 10, Loss: 2017.4067, Accuracy: 0.9024


In [16]:
testing(student)

Test Accuracy of the network on the 10000 test images: 88.52 %


In [ ]:
# https://arxiv.org/abs/1412.6550
# https://www.cs.toronto.edu/~hinton/absps/distillation.pdf